# The five fields and seven guards

Standing needs two things at once: a five-field packet, and seven halt guards *derived from evidence*. One missing field, or one failed guard, is enough to refuse.

The seven guards are not seven booleans you are allowed to tick. They are computed from a finite evidence record: did the pipeline accept, did replay match, is leftover mass zero, was the skeleton preserved, were all declared candidates checked, are lifts pending, is a refusal active.

Run from the package root, or from this `jupyter_notebooks/` folder.

In [1]:
from pathlib import Path
import sys

p = Path.cwd().resolve()
if p.name == "jupyter_notebooks":
    root = p.parent
elif (p / "code" / "coda_fda_core.py").is_file():
    root = p
elif (p / "coda_atlas" / "F005_coda" / "fda" / "code").is_dir():
    root = p / "coda_atlas" / "F005_coda" / "fda"
elif (p / "fda" / "code").is_dir():
    root = p / "fda"
else:
    root = p
assert (root / "code").is_dir(), "run from the package root or jupyter_notebooks/"
if str(root / "code") not in sys.path:
    sys.path.insert(0, str(root / "code"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from sympy import Rational
from coda_fda_core import (
    DescentPacket,
    GuardEvidence,
    derive_guards,
    evaluate_authorization,
)

## An empty carrier is not a packet

The five fields are $K$, $E$, $V$, $\tau$, $\Lambda$. Leave $K$ blank and the record is invalid. Green evidence does not repair a missing carrier.

In [2]:
def green(**overrides):
    base = dict(
        pipeline_accepted=True,
        replay_deterministic=True,
        residual_mass=Rational(0, 1),
        residual_zero_tol=Rational(0, 1),
        skeleton_preserved=True,
        candidates_checked=2,
        candidates_declared=2,
        pending_lifts=0,
        active_refusal=False,
    )
    base.update(overrides)
    if "checked_candidate_ids" not in overrides:
        base["checked_candidate_ids"] = tuple(("candidate_a", "candidate_b")[:base["candidates_checked"]])
    return GuardEvidence(**base)

bad = DescentPacket(
    carrier_k="",
    evaluator_class_e=["candidate_a"],
    verifier_v="v",
    terminalizer_tau="t",
    discipline_lambda="L",
)
print("valid?", bad.is_valid())
refused = evaluate_authorization("proposal_empty_k", bad, green())
print("inhabited:", refused.inhabited, "refusal:", refused.refusal_code)
assert bad.is_valid() is False
assert refused.refusal_code == "R_nopacket"

valid? False
inhabited: False refusal: R_nopacket


## Guards are derived

Start from green evidence. The derived seven-tuple is all true, so the conjunction holds. Checking one of two declared candidates makes `exhaustive` false, and the conjunction fails. Checking zero of zero is also not exhaustive.

In [3]:
ok_guards = derive_guards(green())
print("green conjunction:", ok_guards.is_auth())
assert ok_guards.is_auth() is True

partial = derive_guards(green(candidates_checked=1, candidates_declared=2))
print("partial coverage exhaustive?", partial.exhaustive)
assert partial.exhaustive is False
assert partial.is_auth() is False

vacuous = derive_guards(green(candidates_checked=0, candidates_declared=0))
print("zero of zero exhaustive?", vacuous.exhaustive)
assert vacuous.exhaustive is False

green conjunction: True
partial coverage exhaustive? False
zero of zero exhaustive? False


## Flip any one guard

A valid packet plus green evidence inhabits standing. Mutate any single evidence field so that its derived guard fails, and standing dies. That is the conjunction: all seven, not six of seven.

In [4]:
packet = DescentPacket(
    carrier_k="toy_carrier",
    evaluator_class_e=["candidate_a", "candidate_b"],
    verifier_v="replay_v1",
    terminalizer_tau="tau_standing",
    discipline_lambda="no_self_promote",
)
stood = evaluate_authorization("proposal_ok", packet, green())
print("all green inhabited:", stood.inhabited)
assert stood.inhabited is True

mutations = [
    ("accepted", dict(pipeline_accepted=False)),
    ("deterministic", dict(replay_deterministic=False)),
    ("zero residual", dict(residual_mass=Rational(1, 1))),
    ("skeleton", dict(skeleton_preserved=False)),
    ("exhaustive", dict(candidates_checked=1, candidates_declared=2)),
    ("pending lift", dict(pending_lifts=1)),
    ("refusal", dict(active_refusal=True)),
]
for name, mut in mutations:
    result = evaluate_authorization("proposal_mut", packet, green(**mut))
    print(f"fail {name}: inhabited={result.inhabited} refusal={result.refusal_code}")
    assert result.inhabited is False
    assert result.refusal_code in ("R_candidate", "R_unreplayable")

all green inhabited: True
fail accepted: inhabited=False refusal=R_candidate
fail deterministic: inhabited=False refusal=R_unreplayable
fail zero residual: inhabited=False refusal=R_candidate
fail skeleton: inhabited=False refusal=R_candidate
fail exhaustive: inhabited=False refusal=R_candidate
fail pending lift: inhabited=False refusal=R_candidate
fail refusal: inhabited=False refusal=R_candidate
